#Q1

Existing Code Snippet
```
import java.util.Scanner;

/**
 * The Shape class defines a shape specified in double precision.
 *
 */
public class Shape {
	/**
	 * The shape's immutable width.
	 */
	private final double width;

	private int type;
	static final int SQUARE = 1;
	static final int CIRCLE = 2;
	static final int EQUILATERAL_TRIANGLE = 3;

	/**
	 * Constructs a new Shape with the specified width
	 *
	 * @param type the type of the shape (SQUARE/CIRCLE/EQUILATERAL_TRIANGLE).
	 * @param width the width of the shape
	 */
	public Shape(int type, double width) {
		this.type = type;
		this.width = width;
	}

	/**
	 * Returns the width of this Shape in double precision.
	 *
	 * @return the width of this Shape.
	 */
	public double getWidth() {
		return width;
	}

	/**
	 * Returns the area of this Shape in double precision.
	 *
	 * @return the area of this Shape.
	 */
	public double getArea() {
		switch (type) {
		case SQUARE:
			return width * width;
		case CIRCLE:
			return Math.PI * (width/2) * (width/2);
		case EQUILATERAL_TRIANGLE:
			return (Math.sqrt(3)/4) * width * width;
		default:
			throw new RuntimeException("Unknown shape type encountered: " + type);
		}
	}

	/**
	 * Main method that tests the calculation of Shape properties.
	 *
	 * @param args the unused command line arguments.
	 */
	public static void main(String[] args) {
		System.out.println("-----------------------------------------------------------------");
		System.out.println("ShapeCalc v1.0");
		System.out.println();
		System.out.println("Calculates and prints information for a user-supplied shape and width");
		System.out.println("------------------------------------------------------------------");

		Scanner in = new Scanner(System.in);

	System.out.print("Do you want a square (1), circle (2), or equilateral triangle (3): ");
		int type = in.nextInt();

	System.out.print("Enter the shape's primary dimension (width, radius, or base): ");
		double width = in.nextDouble();

		Shape s = new Shape(type, width);

		System.out.println();
		System.out.println("Shape properties:");
		System.out.println("\tWidth: " + s.getWidth());
		System.out.println("\tArea: " + s.getArea());
	}
}

```

Code Smell:
- Switch Statements – The getArea() method uses a switch statement to calculate the area based on the shape type.
- Adding a new shape requires modifying the existing switch statement, making the code difficult to maintain.

Design Pattern: Strategy Pattern

Refactoring Techniques:
1. Replace Type Code with Subclasses – Replace the integer shape type (SQUARE, CIRCLE, EQUILATERAL_TRIANGLE) with separate subclasses.
2. Replace Conditional with Polymorphism – Move the area calculation logic into individual shape classes.
3. Extract Superclass – Introduce an abstract Shape class to hold common attributes (width) and methods (get_width()).

In [1]:
from abc import ABC, abstractmethod
import math

# Abstract Base Class
class Shape(ABC):
    def __init__(self, width):
        self.width = width
    def get_width(self):
        return self.width
    @abstractmethod
    def get_area(self):
        pass

# Square class
class Square(Shape):
    def get_area(self):
        return self.width * self.width

# Circle class
class Circle(Shape):
    def get_area(self):
        radius = self.width / 2
        return math.pi * radius * radius

# Equilateral Triangle class
class EquilateralTriangle(Shape):
    def get_area(self):
        return (math.sqrt(3) / 4) * self.width * self.width


# Main program
print("Shape Calculator")
print("1. Square")
print("2. Circle")
print("3. Equilateral Triangle")

choice = int(input("Enter your choice: "))
width = float(input("Enter width: "))

if choice == 1:
    shape = Square(width)
elif choice == 2:
    shape = Circle(width)
elif choice == 3:
    shape = EquilateralTriangle(width)
else:
    raise ValueError("Invalid shape type")

print("\nShape properties:")
print("Width:", shape.get_width())
print("Area:", shape.get_area())

Shape Calculator
1. Square
2. Circle
3. Equilateral Triangle
Enter your choice: 1
Enter width: 4

Shape properties:
Width: 4.0
Area: 16.0


#Q2

Existing code snippet

View.java
```
package gui;
import model.Rectangle;


public class View {
  public Rectangle rectangle;
  public void redraw() {
    System.out.println(rectangle);
  }
}
```

Rectangle.java
```
package model;
import static java.lang.Math.*;
import gui.View;


public class Rectangle {
  private Interval horizontal;
  private Interval vertical;
  private View view;


  public Rectangle(View view, int startX, int startY, int endX, int endY) {
    if ((startX > endX) || (startY > endY))
      throw new IllegalArgumentException("Parameters in wrong order.");
    this.horizontal = new Interval(startX, endX);
    this.vertical = new Interval(startY, endY);
    this.view = view; view.rectangle = this;
  }


  /**
   * Create an empty Rectangle with no location.
   */
  public Rectangle(View view) {
    this.horizontal = null;
    this.vertical = null;
    this.view = view; view.rectangle = this;
  }

  public Vertex getCenter() {
    if (horizontal == null)
      return null;
    return new Vertex((horizontal.start + horizontal.end) / 2,
        (vertical.start + vertical.end) / 2);
  }


  public boolean contains(Rectangle parameter) {
    if (parameter.horizontal == null)
      return false;
    if (horizontal == null)
      return false;
    // check whether the horizontal interval
    // contains the horizontal interval
    // in the other rectangle
    // AND
    // check whether the vertical interval
    // contains the vertical interval
    // in the other rectangle
    return ((horizontal.start <= parameter.horizontal.start)
      && (parameter.horizontal.end <= horizontal.end))
      && ((vertical.start <= parameter.vertical.start)
      && (parameter.vertical.end <= vertical.end));
  }


  /**
   * Enlarge the Rectangle just enough so that it contains the other
   * Rectangle. If the Rectangle was empty it gets the same extension
   * as the other Rectangle.
   */
  public void accommodate(Rectangle other) {
    if ((other == null) || (other.horizontal == null))
      return;
    if (horizontal == null) {
      horizontal = new Interval(other.horizontal);
      vertical = new Interval(other.vertical);
      return;
    }
    horizontal.start = min(other.horizontal.start, horizontal.start);
    horizontal.end = max(other.horizontal.end, horizontal.end);
    vertical.start = min(other.vertical.start, vertical.start);
    vertical.end = max(other.vertical.end, vertical.end);
    view.redraw();
  }


  public boolean equals(Object obj) {
    if (!(obj instanceof Rectangle))
      return false;
    Rectangle other = (Rectangle) obj;
    if (horizontal == null)
      return (other.horizontal == null);
    return (horizontal.equals(other.horizontal))
        && (vertical.equals(other.vertical));
  }


  public String toString() {
    return "Rectangle(" + horizontal + " x " + vertical + ")";
  }
}
```

Code Smell:
- Inappropriate Intimacy – The Rectangle class directly accesses and modifies the View class.
- This creates tight coupling between the Model and View.

Design Pattern: Observer Pattern.

Refactoring Techniques:
1. Replace Bidirectional Association with Unidirectional Association – Remove the direct bidirectional dependency between Rectangle and View.
2. Extract Method – Separate the notification logic into a dedicated notify_observers() method.
3. Move Method – Move the redraw responsibility to the View class, allowing it to handle its own display operations.

In [3]:
from abc import ABC, abstractmethod

# Abstract Observer class
class Observer(ABC):
    @abstractmethod
    def update(self):
        pass

# View class
class View(Observer):
    def update(self, rectangle):
        self.redraw(rectangle)
    def redraw(self, rectangle):
        print("Rectangle Updated:", rectangle)

# Interval class
class Interval:
    def __init__(self, start, end):
        self.start = start
        self.end = end
    def copy(self):
        return Interval(self.start, self.end)
    def __str__(self):
        return f"[{self.start}, {self.end}]"

# Rectangle class (Subject)
class Rectangle:
    def __init__(self, startX=None, startY=None,endX=None, endY=None):
        self.observers = []
        if startX is not None:
            if startX > endX or startY > endY:
                raise ValueError("Parameters in wrong order")
            self.horizontal = Interval(startX, endX)
            self.vertical = Interval(startY, endY)
        else:
            self.horizontal = None
            self.vertical = None

    def add_observer(self, observer):
        self.observers.append(observer)

    def notify_observers(self):
        for observer in self.observers:
            observer.update(self)

    def accommodate(self, other):
        if other is None or other.horizontal is None:
            return
        if self.horizontal is None:
            self.horizontal = other.horizontal.copy()
            self.vertical = other.vertical.copy()
        else:
            self.horizontal.start = min(self.horizontal.start, other.horizontal.start)
            self.horizontal.end = max(self.horizontal.end, other.horizontal.end)
            self.vertical.start = min(self.vertical.start, other.vertical.start)
            self.vertical.end = max(self.vertical.end, other.vertical.end)
        self.notify_observers()

    def __str__(self):
        return f"Rectangle({self.horizontal} x {self.vertical})"


# Main program
view = View()

rectangle = Rectangle(10, 10, 20, 20)
rectangle.add_observer(view)

other = Rectangle(5, 5, 25, 25)

rectangle.accommodate(other)

Rectangle Updated: Rectangle([5, 25] x [5, 25])
